### Normalize and correct sgRNA Log Fold Change data
#### adapted from Fortin et al, 2019
#### by Stefanus Bernard

In [ ]:
import pandas as pd
import numpy as np
from normalize_lfc_utils import *

In [ ]:
read_count = pd.read_csv("../../data/sgrna_lfc_data/tkov3_data/matrix-reads-by-gRNA-RPE1-drugZ.txt", sep ='\t')
read_count[['gene', 'sgRNA']] = read_count['sgRNA'].str.split('_', n = 1, expand=True)

# rename gene and spacer columns
read_count = read_count.drop(columns=['GENE', 'gene'])
read_count = read_count.rename(columns={'sgRNA':'spacer'})

# filtered out sgRNA with less than 30 reads in T0
read_count = read_count[read_count['RPE1_T0'] > 30]

# drop LOPRIMER data
read_count = read_count[['spacer', 'RPE1_T0', 'RPE1_T3A_CTRL', 'RPE1_T3B_CTRL']]
display(read_count)

In [ ]:
# import library data
library_data = pd.read_csv("../../public_crispr_library/restricted_library/tkov3_guide_sequence.tsv", sep ="\t", header = None)
library_data.columns = ['sgRNA', 'spacer', 'gene']
display(library_data.shape)

In [ ]:
read_count = pd.merge(read_count, library_data, how="left", on="spacer").set_index(['sgRNA', 'spacer', 'gene']).sort_values(by="sgRNA")
read_count

In [ ]:
# we log transformed the raw read counts (log2(counts + 1))

# separate plasmid count into a new column
# plasmid_count = read_count['RPE1_T0']
# plasmid_count

# main log transformed read count
# log_transformed_read_count = read_count.drop(columns=['RPE1_T0'])
log_transformed_read_count = np.log2(read_count + 1)
log_transformed_read_count

In [ ]:
RPE1_T0 = log_transformed_read_count[['RPE1_T0']]

# Compute log-fold change relative to T0
sgrna_lfc = log_transformed_read_count.drop(columns=['RPE1_T0']).subtract(RPE1_T0.values, axis = 0)
sgrna_lfc

In [ ]:
# Center LFCs using non-essential guides
non_essential_mask = ~log_transformed_read_count.index.get_level_values(2).isin(['LacZ', 'luciferase', 'EGFP'])
median_non_essential = sgrna_lfc[non_essential_mask].median(axis=0)

median_non_essential

In [ ]:
# Subtract median of non-essential guides for each sample (normalized sgrna LFC)
sgrna_lfc_normalized = sgrna_lfc - median_non_essential
sgrna_lfc_normalized

In [ ]:
# scaled sgRNA log fold change by centering around the median LFC of guides targeting essential genes
norm_sgrna_lfc_scaled, list_common_essential = scale_essential(sgrna_lfc_normalized, '../../data/sgrna_lfc_data/constitutive_core_essential_hart_2014.csv')
norm_sgrna_lfc_scaled

In [ ]:
# check whether the normalization works (the median of lfc data should be around 0)
sanity_check_scale_essential(sgrna_lfc, norm_sgrna_lfc_scaled, list_common_essential)

In [ ]:
# averaged sgRNA LFC across replicates
lfc_norm_scaled_avg = mean_row(norm_sgrna_lfc_scaled)
lfc_norm_scaled_avg = lfc_norm_scaled_avg.reset_index()
lfc_norm_scaled_avg

In [ ]:
lfc_norm_scaled_avg.to_csv("../../data/sgrna_lfc_data/output_normalized/tkov3_rpe1_normalized_lfc.csv", index=False)